## Processing Files: Mini-Project ##

In [36]:
import h5py
import numpy as np
import os 
import pandas as pd
import matplotlib.pyplot as plt 
from matplotlib.colors import LogNorm
from sklearn.utils import shuffle


import math

# TensorFlow and tf.keras
import tensorflow as tf
from tensorflow import keras

import matplotlib.style
import matplotlib as mpl 
from scipy import linalg


#Set default figure size
#mpl.rcParams['figure.figsize'] = [12.0, 8.0] #Inches... of course it is inches
mpl.rcParams["legend.frameon"] = False
mpl.rcParams['figure.dpi']=200 # dots per inch

#Useful for debugging problems
print(tf.__version__)
print(np.__version__)


2.13.1
1.24.3


In [37]:
def lognorm_transform(img, vmin=0.01, vmax=1e3):
    '''
    Log-normalises jet images to [0, 1] range.
    Clips values below vmin to vmin before taking log,
    avoiding -inf and the need for a separate masking step.
    '''
    log_min = np.log(vmin)
    log_max = np.log(vmax)

    img_clipped = np.clip(img, vmin, vmax).astype(np.float32)
    img_log     = np.log(img_clipped)

    return (img_log - log_min) / (log_max - log_min)

In [38]:
def load_single_h5_img(file_path, channels='combined'):
    '''
    Loads individual jet images from one h5 file.
    Applies lognorm_transform immediately so the returned
    array is already float32 in [0,1] — no second pass needed.

    Returns
    -------
    images     : float32 array, shape (N, 100, 100) or (N, 100, 100, 3)
    int_labels : int array,     shape (N,)   — 0=Gluon 1=Quark 2=W 3=Z 4=Top
    '''
    try:
        df = h5py.File(file_path, 'r')
    except Exception as e:
        print("Error opening file:", e)
        raise

    # Integer class labels from one-hot columns
    feat_names = [l.decode('utf-8') for l in df['jetFeatureNames'][:]]
    pJet       = pd.DataFrame(np.array(df['jets']), columns=feat_names)
    label_cols = ['j_g', 'j_q', 'j_w', 'j_z', 'j_t']
    int_labels = np.argmax(pJet[label_cols].to_numpy(), axis=1)

    # Load images and transform immediately — before appending to the list
    if channels == 'combined':
        raw    = np.array(df['jetImage'][:])
        images = lognorm_transform(raw)                        # (N, 100, 100)

    elif channels == 'all':
        ch0 = lognorm_transform(np.array(df['jetImage'][:]))
        ch1 = lognorm_transform(np.array(df['jetImageECAL'][:]))
        ch2 = lognorm_transform(np.array(df['jetImageHCAL'][:]))
        images = np.stack([ch0, ch1, ch2], axis=-1)           # (N, 100, 100, 3)

    else:
        raise ValueError(f"channels must be 'combined' or 'all', got '{channels}'")

    df.close()
    return images, int_labels
    

In [39]:
def load_many_h5_files(main_dir, max_files=None, dataset_type="train",
                       save_dir=".", channels='combined'):
    '''
    Loads all .h5 files in a directory and saves combined arrays as .npy files.

    Parameters
    ----------
    main_dir : str
        Directory containing .h5 files.
    max_files : int or None
        Limit number of files loaded. None loads all.
    dataset_type : str
        Used in output filenames e.g. "train", "validation", "test".
    save_dir : str
        Directory where .npy files are saved.
    channels : str
        Passed through to load_single_h5_img.
        'combined' → single jetImage channel  (Task 1)
        'all'      → three calorimeter channels (Extension 1)

    Returns
    -------
    all_images : np.ndarray   shape (N_total, 100, 100) or (N_total, 100, 100, 3)
    all_labels : np.ndarray   shape (N_total,)  integer class labels 0–4
    '''

    all_images = []
    all_labels = []
    file_count = 0

    h5_files = sorted([f for f in os.listdir(main_dir) if f.endswith('.h5')])

    for file in h5_files:

        if max_files is not None and file_count >= max_files:
            break

        file_path = os.path.join(main_dir, file)

        try:
            images, labels = load_single_h5_img(file_path, channels=channels)
            all_images.append(images)
            all_labels.append(labels)
            file_count += 1
            print(f"Loaded {file_path}  ({images.shape[0]:,} jets)")

        except Exception as e:
            print(f"Skipping {file_path}: {e}")
            continue

    all_images = np.concatenate(all_images, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)

    # Verify class balance before saving
    unique, counts = np.unique(all_labels, return_counts=True)
    class_names = ['Gluon', 'Quark', 'W', 'Z', 'Top']
    print("\nClass distribution:")
    for cls, cnt in zip(unique, counts):
        print(f"  {class_names[cls]:6s}: {cnt:,}")

    # Save
    images_file = os.path.join(save_dir, f"{dataset_type}_images.npy")
    labels_file = os.path.join(save_dir, f"{dataset_type}_labels.npy")
    np.save(images_file, all_images)
    np.save(labels_file, all_labels)

    print(f"\nSaved {all_images.shape} → {images_file}")
    print(f"Saved {all_labels.shape} → {labels_file}")
    print(f"Total: {file_count} files, {all_images.shape[0]:,} jets")

    return all_images, all_labels

In [40]:
#Directories
save_dir = './datasets/'
validation_dir = './data_files/val/'
train_dir = './data_files/train/'

In [47]:
validation_combined_images, validation_combined_labels = load_many_h5_files(main_dir=validation_dir, 
                                                                            max_files= 3,
                                                                            dataset_type = "validation", 
                                                                           save_dir=save_dir)

Loaded ./data_files/val/jetImage_7_100p_0_10000.h5  (10,000 jets)
Loaded ./data_files/val/jetImage_7_100p_10000_20000.h5  (10,000 jets)
Loaded ./data_files/val/jetImage_7_100p_30000_40000.h5  (10,000 jets)

Class distribution:
  Gluon : 6,083
  Quark : 5,911
  W     : 6,006
  Z     : 5,975
  Top   : 6,025

Saved (30000, 100, 100) → ./datasets/validation_images.npy
Saved (30000,) → ./datasets/validation_labels.npy
Total: 3 files, 30,000 jets


In [48]:
validation_combined_images.shape

(30000, 100, 100)

In [49]:
train_combined_images, train_combined_labels = load_many_h5_files(main_dir = train_dir, 
                                                                 max_files = 8, 
                                                                 dataset_type = 'train', 
                                                                 save_dir = save_dir)

Loaded ./data_files/train/jetImage_0_100p_0_10000.h5  (10,000 jets)
Loaded ./data_files/train/jetImage_0_100p_10000_20000.h5  (10,000 jets)
Loaded ./data_files/train/jetImage_0_100p_20000_30000.h5  (10,000 jets)
Loaded ./data_files/train/jetImage_0_100p_30000_40000.h5  (10,000 jets)
Loaded ./data_files/train/jetImage_0_100p_40000_50000.h5  (10,000 jets)
Loaded ./data_files/train/jetImage_0_100p_50000_60000.h5  (10,000 jets)
Loaded ./data_files/train/jetImage_0_100p_60000_70000.h5  (10,000 jets)
Loaded ./data_files/train/jetImage_0_100p_70000_80000.h5  (10,000 jets)

Class distribution:
  Gluon : 16,193
  Quark : 15,477
  W     : 16,027
  Z     : 15,980
  Top   : 16,323

Saved (80000, 100, 100) → ./datasets/train_images.npy
Saved (80000,) → ./datasets/train_labels.npy
Total: 8 files, 80,000 jets


In [50]:
train_combined_images.shape

(80000, 100, 100)

# Averaged Image data files: #

In [3]:
def load_single_avg_h5_img(file_path):
    
    '''
    Loads a single h5 image file and produces average image ouputs for each classifier
    
    Labels:
    0 = j_g
    1 = j_q
    2 = j_w
    3 = j_z
    4 = j_t
    
    '''
    
    
    try:
        df = h5py.File(file_path,'r') ## opening file
        
    except Exception as e:
        print("Error:", e)
        
    #Labels and converting df into a pandas dataframe
    
    labels=[label.decode("utf-8") for label in df['jetFeatureNames'][:] ]
    
    #dataframe of values per jet
    pJet = pd.DataFrame(np.array(df['jets']),columns=labels)
    
    averageGluon=np.mean(df['jetImage']*np.array(pJet['j_g'])[:,None, None], axis =0)
    averageQuark=np.mean(df['jetImage'][:]*np.array(pJet['j_q'])[:,None,None],axis=0)
    averageW=np.mean(df['jetImage'][:]*np.array(pJet['j_w'])[:,None,None],axis=0)
    averageZ=np.mean(df['jetImage'][:]*np.array(pJet['j_z'])[:,None,None],axis=0)
    averageT=np.mean(df['jetImage'][:]*np.array(pJet['j_t'])[:,None,None],axis=0)
    averageAll=np.mean(df['jetImage'][:],axis=0)
    
    images = np.array([averageGluon, averageQuark, averageW, averageZ, averageT])
    labels = np.array([0,1,2,3,4])

    
    return images, labels

In [4]:
def load_many_avg_h5_files(main_dir, max_files=None, dataset_type="train", save_dir="."):
    '''
    Loads h5 files from a directory and saves combined arrays as .npy files.

    Parameters
    ----------
    main_dir : str
        Directory containing .h5 files

    max_files : int or None
        Limit number of files loaded. None loads all files.

    dataset_type : str
        "train" or "validation" (used in output filenames)

    save_dir : str
        Directory where .npy files will be saved
    '''

    all_images = []
    all_labels = []
    file_count = 0

    for file in os.listdir(main_dir):
        try:
            
            if file.endswith(".h5"):

                if max_files is not None and file_count >= max_files:
                    break

                file_path = os.path.join(main_dir, file)

                images, labels = load_single_h5_img(file_path)

                all_images.append(images)
                all_labels.append(labels)

                file_count += 1
                print("FILE PROCESSED:", file_path)
        
        except Exception as e:
            print("ERROR OCCURRED:", e)
            continue

    

    all_images = np.concatenate(all_images, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)

    # Create filenames automatically
    images_file = os.path.join(save_dir, f"{dataset_type}_images.npy")
    labels_file = os.path.join(save_dir, f"{dataset_type}_labels.npy")

    # Save arrays
    np.save(images_file, all_images)
    np.save(labels_file, all_labels)

    print(f"Saved images to: {images_file}")
    print(f"Saved labels to: {labels_file}")
    print(f"All files processed ({file_count} files)")

    return all_images, all_labels

# Loading validation and train datasets: #

In [5]:
#Loading all validation data files

validation_dir = './data_files/val/'
train_dir = './data_files/train/'

In [51]:
#val_images, val_labels = load_many_avg_h5_files(validation_dir, dataset_type= 'validation_avg')

In [52]:
#train_images, train_labels = load_many_avg_h5_files(train_dir, dataset_type = 'train_avg')

# Loading Features Data: #

In [61]:
def load_h5_features(main_dir, max_files=None, save_dir = '.', dataset_type = 'train'):
    '''
    Loads jet feature variables from h5 files for Phase 3 analysis.
    Returns images, integer labels, and the full jet features dataframe.
    Does not save any .npy files.
    '''
    all_images   = []
    all_labels   = []
    all_features = []

    h5_files = sorted([f for f in os.listdir(main_dir) if f.endswith('.h5')])
    file_count = 0

    for fname in h5_files:
        if max_files is not None and file_count >= max_files:
            break

        path = os.path.join(main_dir, fname)
        try:
            df = h5py.File(path, 'r')

            feat_names = [l.decode('utf-8') for l in df['jetFeatureNames'][:]]
            pJet       = pd.DataFrame(np.array(df['jets']), columns=feat_names)

            label_cols = ['j_g', 'j_q', 'j_w', 'j_z', 'j_t']
            int_labels = np.argmax(pJet[label_cols].to_numpy(), axis=1)

            images = lognorm_transform(np.array(df['jetImage'][:]))

            all_images.append(images)
            all_labels.append(int_labels)
            all_features.append(pJet)

            df.close()
            file_count += 1
            print(f"Loaded {fname} ({images.shape[0]:,} jets)")

        except Exception as e:
            print(f"Skipped {fname}: {e}")
            continue

    all_images   = np.concatenate(all_images,   axis=0)
    all_labels   = np.concatenate(all_labels,   axis=0)
    all_features = pd.concat(all_features, ignore_index=True)
    
    #Creating directory and file:
    
    images_file = os.path.join(save_dir, f"{dataset_type}_images.npy")
    labels_file = os.path.join(save_dir, f"{dataset_type}_labels.npy")
    
    #Saving files
    
    np.save(images_file, all_images)
    np.save(labels_file, all_labels)
    
    # Save features as .parquet — preserves column names and dtypes
    features_file = os.path.join(save_dir, f"{dataset_type}_features.parquet")
    all_features.to_parquet(features_file, index=False)

    print(f"\nTotal: {all_images.shape[0]:,} jets, {all_features.shape[1]} features")
    return all_images, all_labels, all_features

In [62]:
validation_images_p4, validation_labels_p4, validation_features_p4 = load_h5_features(validation_dir,
                                                                                     max_files = 3, 
                                                                                     dataset_type='val_p4')

Loaded jetImage_7_100p_0_10000.h5 (10,000 jets)
Loaded jetImage_7_100p_10000_20000.h5 (10,000 jets)
Loaded jetImage_7_100p_30000_40000.h5 (10,000 jets)

Total: 30,000 jets, 59 features


In [63]:
data_images_p4, data_labels_p4, data_features_p4 = load_h5_features(train_dir,
                                                                   max_files = 8, 
                                                                   dataset_type= 'data_p4')

Loaded jetImage_0_100p_0_10000.h5 (10,000 jets)
Loaded jetImage_0_100p_10000_20000.h5 (10,000 jets)
Loaded jetImage_0_100p_20000_30000.h5 (10,000 jets)
Loaded jetImage_0_100p_30000_40000.h5 (10,000 jets)
Loaded jetImage_0_100p_40000_50000.h5 (10,000 jets)
Loaded jetImage_0_100p_50000_60000.h5 (10,000 jets)
Loaded jetImage_0_100p_60000_70000.h5 (10,000 jets)
Loaded jetImage_0_100p_70000_80000.h5 (10,000 jets)

Total: 80,000 jets, 59 features


In [60]:
validation_features_p4.shape

(30000, 59)